# 07 — Neural Network Training (Deep Learning Family)
This notebook trains a simple Multi-Layer Perceptron (MLP). **Crucially, we apply StandardScaler here** as distance-based models are sensitive to feature scales, unlike tree models.

**Outputs:**
1. `submissions/nn_submission.csv` — Individual model submission.
2. `pickles/nn_oof.pkl` — Out-Of-Fold probabilities for ensembling.
3. `pickles/nn_test_preds.pkl` — Test probabilities for ensembling.

In [1]:
# Mount Google Drive - run this first on Colab
from google.colab import drive
drive.mount('/content/drive')

# Set your project path on Drive - change this if your folder name is different
DRIVE_PATH = '/content/drive/MyDrive/santander-customer-satisfaction/'
PICKLE_DIR = DRIVE_PATH + 'pickles/'
SUBMIT_DIR = DRIVE_PATH + 'submissions/'

import os
os.makedirs(SUBMIT_DIR, exist_ok=True)
print('Drive mounted. Paths set.')

Mounted at /content/drive
Drive mounted. Paths set.


In [2]:
import pandas as pd
import numpy as np
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

TensorFlow version: 2.20.0


## 1. Load Data & Shared CV Folds

In [3]:
# Load clean datasets from Drive
# train = pd.read_pickle(f'{PICKLE_DIR}train_clean.pkl')
# test = pd.read_pickle(f'{PICKLE_DIR}test_clean.pkl')

train = pd.read_pickle(f'{PICKLE_DIR}train_advanced.pkl')
test = pd.read_pickle(f'{PICKLE_DIR}test_advanced.pkl')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

# Separate features and target
X = train.drop(columns=['TARGET', 'ID'], errors='ignore')
y = train['TARGET']
test_features = test.drop(columns=['ID'], errors='ignore')

# Align columns
test_features = test_features[X.columns]

# Load identical CV folds
with open(f'{PICKLE_DIR}cv_fold_indices.pkl', 'rb') as f:
    cv_folds = pickle.load(f)

print(f"Loaded {len(cv_folds)} CV folds.")

Train shape: (76020, 104)
Test shape: (75818, 103)
Loaded 5 CV folds.


## 2. Train Neural Network (MLP)

In [4]:
# Arrays to store out-of-fold predictions and test predictions
oof_preds = np.zeros(len(train))
test_preds = np.zeros(len(test))

fold_aucs = []

print("Starting Neural Network Training...")
print("-" * 40)

for fold_dict in cv_folds:
    fold = fold_dict['fold']
    train_idx = fold_dict['train_idx']
    val_idx = fold_dict['val_idx']

    # Split data
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    # IMPORTANT: Scaling for Neural Networks
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(test_features)

    # Define Model Architecture
    model = Sequential([
        Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
        BatchNormalization(),
        Dropout(0.2),
        Dense(64, activation='relu'),
        BatchNormalization(),
        Dropout(0.2),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='auc')]
    )

    # Early Stopping
    es = EarlyStopping(
        monitor='val_auc',
        patience=10,
        mode='max',
        restore_best_weights=True,
        verbose=0
    )

    # Train
    model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=100,
        batch_size=512,
        callbacks=[es],
        verbose=0
    )

    # Predict
    val_preds = model.predict(X_val_scaled).flatten()
    oof_preds[val_idx] = val_preds

    # Evaluate
    fold_auc = roc_auc_score(y_val, val_preds)
    fold_aucs.append(fold_auc)
    print(f"Fold {fold} AUC: {fold_auc:.5f}")

    # Predict on Test (accumulate)
    test_preds += model.predict(X_test_scaled).flatten() / len(cv_folds)

print("-" * 40)
print(f"Mean CV AUC: {np.mean(fold_aucs):.5f} ± {np.std(fold_aucs):.5f}")

Starting Neural Network Training...
----------------------------------------


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


476/476 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Fold 0 AUC: 0.81612
2370/2370 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


476/476 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Fold 1 AUC: 0.83164
2370/2370 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step


...

476/476 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Fold 2 AUC: 0.83577
2370/2370 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


476/476 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Fold 3 AUC: 0.82079
2370/2370 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


476/476 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Fold 4 AUC: 0.82914
2370/2370 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step
----------------------------------------
Mean CV AUC: 0.82669 ± 0.00720


## 3. Save OOF & Test Predictions for the Ensemble

In [7]:
# Save OOF predictions
oof_df = pd.DataFrame({'ID': train['ID'], 'nn_pred': oof_preds})
oof_df.to_pickle(f'{PICKLE_DIR}nn_oof.pkl')

# Save Test predictions
test_preds_df = pd.DataFrame({'ID': test['ID'], 'nn_pred': test_preds})
test_preds_df.to_pickle(f'{PICKLE_DIR}nn_test_preds.pkl')

print("✅ Saved Neural Network probabilities to Drive! (Ready for Ensembling!)")

✅ Saved Neural Network probabilities to Drive! (Ready for Ensembling!)


## 4. Kaggle Submission

In [8]:
submission = pd.DataFrame({
    'ID': test['ID'],
    'TARGET': test_preds
})

submission_path = f'{SUBMIT_DIR}nn_submission.csv'
submission.to_csv(submission_path, index=False)

print(f"✅ Kaggle Submission saved to {submission_path}")

✅ Kaggle Submission saved to /content/drive/MyDrive/santander-customer-satisfaction/submissions/nn_submission.csv
